# 🎬 XsuperAnimations — free anime studio on Colab

Prompt / script / rough doodle → **anime episode** with consistent characters, voices, lip flaps, camera work and
sakuga effects. Everything here is **free**: open models on Colab's free GPU.

**How to use:** `Runtime → Change runtime type → T4 GPU`, then run the cells **1 → 4** in order.
Your projects are saved to Google Drive, so a long episode can be rendered over several sessions —
re-running a render only does the shots that aren't finished yet.

> 💡 Free Colab gives a few GPU hours per day. A full 23-minute episode in *final* quality takes several sessions.
> [Kaggle](https://www.kaggle.com/code) notebooks are also free (30 GPU hours/week, more RAM) and run this notebook too.

In [ ]:
#@title 1. Connect Google Drive & install XsuperAnimations (~2 min)
SAVE_TO_DRIVE = True #@param {type:"boolean"}
import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/XsuperAnimations'
else:
    WORK = '/content/XsuperAnimations'
os.makedirs(f'{WORK}/projects', exist_ok=True)
os.makedirs(f'{WORK}/loras', exist_ok=True)

if os.path.exists('/content/xsuper-src'):
    !git -C /content/xsuper-src pull -q
else:
    !git clone -q https://github.com/openaideveloperai-png/XsuperAnimations /content/xsuper-src
!apt-get -qq install -y espeak-ng ffmpeg > /dev/null
!pip install -q -e "/content/xsuper-src[ui]" kokoro soundfile "misaki[ja]"
print('✅ XsuperAnimations installed. Work folder:', WORK)

In [ ]:
#@title 2. Install ComfyUI + free anime models (~10-15 min, once per session)
#@markdown **Art model** (all free & open). Animagine XL 4 is the default and does the clean modern TV look.
ART_MODEL = "Animagine XL 4.0" #@param ["Animagine XL 4.0", "Illustrious XL 1.0", "NoobAI XL 1.1", "Custom URL"]
CUSTOM_URL = "" #@param {type:"string"}
#@markdown Sketch → anime (ControlNet scribble), AI in-between motion (Wan 2.2, ~17 GB), anime upscaler:
SKETCH_MODEL = True #@param {type:"boolean"}
MOTION_MODEL = True #@param {type:"boolean"}

import os, fnmatch, shutil, subprocess, time, requests
if not os.path.exists('/content/ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
    !pip install -q -r /content/ComfyUI/requirements.txt
from huggingface_hub import hf_hub_download, list_repo_files
M = '/content/ComfyUI/models'

def hf(repo, patterns, folder, rename=None):
    os.makedirs(f'{M}/{folder}', exist_ok=True)
    files = list_repo_files(repo)
    for pat in patterns:
        hits = [f for f in files if fnmatch.fnmatch(f, pat)]
        if hits:
            break
    else:
        raise RuntimeError(f'No file like {patterns} in {repo}. Files: {files[:15]}')
    name = rename or os.path.basename(hits[0])
    dest = f'{M}/{folder}/{name}'
    if not os.path.exists(dest):
        print('⬇️ ', repo, hits[0])
        shutil.move(hf_hub_download(repo, hits[0], local_dir='/content/hf_tmp'), dest)
    return name

ART = {
    "Animagine XL 4.0": ("cagliostrolab/animagine-xl-4.0", ["animagine-xl-4.0-opt.safetensors", "animagine-xl-4.0.safetensors", "*.safetensors"], 1),
    "Illustrious XL 1.0": ("OnomaAIResearch/Illustrious-XL-v1.0", ["Illustrious-XL-v1.0.safetensors", "*.safetensors"], 2),
    "NoobAI XL 1.1": ("Laxhar/noobai-XL-1.1", ["NoobAI-XL-v1.1.safetensors", "*.safetensors"], 2),
}
if ART_MODEL == "Custom URL":
    assert CUSTOM_URL, "Paste a direct .safetensors download link into CUSTOM_URL"
    CHECKPOINT, CLIP_SKIP = 'custom_anime.safetensors', 2
    os.makedirs(f'{M}/checkpoints', exist_ok=True)
    !wget -q -c -O "{M}/checkpoints/{CHECKPOINT}" "{CUSTOM_URL}"
else:
    repo, pats, CLIP_SKIP = ART[ART_MODEL]
    CHECKPOINT = hf(repo, [p for p in pats], 'checkpoints')
if SKETCH_MODEL:
    hf('xinsir/controlnet-scribble-sdxl-1.0', ['diffusion_pytorch_model.safetensors', '*.safetensors'],
       'controlnet', 'controlnet-scribble-sdxl-1.0.safetensors')
if MOTION_MODEL:
    hf('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', ['split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors', '*wan2.2_ti2v_5B*.safetensors'], 'diffusion_models')
    hf('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', ['split_files/vae/wan2.2_vae.safetensors', '*wan2.2_vae*.safetensors'], 'vae')
    hf('Comfy-Org/Wan_2.1_ComfyUI_repackaged', ['split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', '*umt5_xxl_fp8*.safetensors'], 'text_encoders')
    os.makedirs(f'{M}/upscale_models', exist_ok=True)
    !wget -q -c -O {M}/upscale_models/RealESRGAN_x4plus_anime_6B.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth

# character LoRAs live on Drive so they survive between sessions
open('/content/ComfyUI/extra_model_paths.yaml', 'w').write(f'xsuper:\n  base_path: {WORK}\n  loras: loras\n')

# (re)start ComfyUI in the background
subprocess.run('pkill -f "ComfyUI/main.py"', shell=True)
comfy = subprocess.Popen(['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188',
                          '--preview-method', 'none'], stdout=open('/content/comfyui.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        requests.get('http://127.0.0.1:8188/system_stats', timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('ComfyUI did not start - see /content/comfyui.log')
print(f'✅ ComfyUI running with {CHECKPOINT}')

In [ ]:
#@title 3. Open a project (new projects start with the demo episode)
PROJECT = "my_anime" #@param {type:"string"}
import os, yaml
PROJECT_DIR = f'{WORK}/projects/{PROJECT}'
if not os.path.exists(f'{PROJECT_DIR}/episode.yaml'):
    !xsuper new "{PROJECT_DIR}" --example
# point the project at the models installed in step 2
cfg_path = f'{PROJECT_DIR}/studio.yaml'
cfg = yaml.safe_load(open(cfg_path)) or {}
cfg.setdefault('image', {}).update(checkpoint=CHECKPOINT, clip_skip=CLIP_SKIP)
cfg.setdefault('video', {})['enabled'] = bool(MOTION_MODEL)
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)
!xsuper check "{PROJECT_DIR}"
!xsuper doctor "{PROJECT_DIR}"

In [ ]:
#@title 4. Launch the studio app (click the *public URL* it prints)
#@markdown Story → Script → Characters → Doodle → Render, all in the browser. Stop this cell to use the cells below.
!xsuper ui "{PROJECT_DIR}" --share

---
### Optional: render from here instead of the app

In [ ]:
#@title Render (animatic = free preview · draft = fast · final = full quality)
MODE = "draft" #@param ["animatic", "draft", "final"]
ONLY_SHOTS = "" #@param {type:"string"}
REROLL_SHOTS = "" #@param {type:"string"}
args = f'--mode {MODE}' + (f' --only {ONLY_SHOTS}' if ONLY_SHOTS else '') + (f' --reroll {REROLL_SHOTS}' if REROLL_SHOTS else '')
!xsuper render "{PROJECT_DIR}" {args}

### Optional (advanced): lock a character's design with a LoRA
1. Draw a model sheet (below or in the app's *Characters* tab).
2. Open `characters/<name>/dataset` on Drive and delete drawings that are off-model.
3. Train (~45-90 min on a T4). The LoRA is saved to `XsuperAnimations/loras` on Drive.
4. In `episode.yaml` add `lora: <name>.safetensors` and `trigger: <name>_xs` to that character.

In [ ]:
#@title Draw model sheet + train character LoRA
CHARACTER = "ren" #@param {type:"string"}
DRAWINGS = 24 #@param {type:"integer"}
TRAIN_NOW = False #@param {type:"boolean"}
!xsuper sheet "{PROJECT_DIR}" {CHARACTER} --count {DRAWINGS}
if TRAIN_NOW:
    if not os.path.exists('/content/sd-scripts'):
        !git clone -q https://github.com/kohya-ss/sd-scripts /content/sd-scripts
        !cd /content/sd-scripts && pip install -q -r requirements.txt bitsandbytes
    !cd /content/sd-scripts && SD_SCRIPTS=/content/sd-scripts CHECKPOINT="{M}/checkpoints/{CHECKPOINT}" OUT="{WORK}/loras" bash "{PROJECT_DIR}/characters/{CHARACTER}/train_lora.sh"